In [1]:
import numpy as np
import pandas as pd
from math import sqrt, exp

In [6]:
# American option with continuous dividend yield via Cox-Ross-Rubinstein binomial tree
def american_option_binomial(option_type, S, K, T, r, q, sigma, steps=500):
    dt = T / steps
    u = exp(sigma * sqrt(dt))
    d = 1.0 / u
    p = (exp((r - q) * dt) - d) / (u - d)
    disc = exp(-r * dt)
    
    # stock prices at maturity
    j = np.arange(steps + 1)
    stock = S * (u ** j) * (d ** (steps - j))

    # option payoff at maturity
    if option_type.lower() == "call":
        values = np.maximum(stock - K, 0.0)
    else:
        values = np.maximum(K - stock, 0.0)

    # backward induction with early exercise
    for i in range(steps - 1, -1, -1):
        continuation = disc * (p * values[1:] + (1 - p) * values[:-1])

        j = np.arange(i + 1)
        stock_i = S * (u ** j) * (d ** (i - j))

        if option_type.lower() == "call":
            exercise = np.maximum(stock_i - K, 0.0)
        else:
            exercise = np.maximum(K - stock_i, 0.0)

        values = np.maximum(continuation, exercise)

    return float(values[0])

def american_option_with_greeks(option_type, S, K, T, r, q, sigma, steps=500):
    value = american_option_binomial(option_type, S, K, T, r, q, sigma, steps)

    # Delta: uses central difference on stock with tiny bumps
    dS_delta = 0.001
    value_up_delta = american_option_binomial(option_type, S + dS_delta, K, T, r, q, sigma, steps)
    value_down_delta = american_option_binomial(option_type, S - dS_delta, K, T, r, q, sigma, steps)
    delta = (value_up_delta - value_down_delta) / (2.0 * dS_delta)

    # Gamma
    dS_gamma = 1.5
    value_up = american_option_binomial(option_type, S + dS_gamma, K, T, r, q, sigma, steps)
    value_down = american_option_binomial(option_type, S - dS_gamma, K, T, r, q, sigma, steps)
    gamma = (value_up - 2.0 * value + value_down) / (dS_gamma ** 2)

    # Vega
    dvol = 0.001
    value_vol_up = american_option_binomial(option_type, S, K, T, r, q, sigma + dvol, steps)
    value_vol_down = american_option_binomial(option_type, S, K, T, r, q, sigma - dvol, steps)
    vega = (value_vol_up - value_vol_down) / (2.0 * dvol)

    # Rho: bump r and q together to keep (r - q) fixed
    dr = 0.001
    value_up_r = american_option_binomial(option_type, S, K, T, r + dr, q + dr, sigma, steps)
    value_down_r = american_option_binomial(option_type, S, K, T, r - dr, q - dr, sigma, steps)
    rho = (value_up_r - value_down_r) / (2.0 * dr)

    # Theta: positive time decay convention
    dT = 1.0 / 365.0
    if T > dT:
        Vt_up = american_option_binomial(option_type, S, K, T + dT, r, q, sigma, steps)
        Vt_down = american_option_binomial(option_type, S, K, T - dT, r, q, sigma, steps)
        theta = (Vt_up - Vt_down) / (2.0 * dT)
    else:
        theta = np.nan

    return value, delta, gamma, vega, rho, theta


df = pd.read_csv("/Users/fuyuxuan/Downloads/test12_1.csv")

# Remove blank rows
df = df.dropna(how="all").copy()
results = []

for _, row in df.iterrows():
    option_id = int(row["ID"])
    option_type = row["Option Type"]
    S = float(row["Underlying"])
    K = float(row["Strike"])
    T = float(row["DaysToMaturity"]) / float(row["DayPerYear"])
    r = float(row["RiskFreeRate"])
    q = float(row["DividendRate"])
    sigma = float(row["ImpliedVol"])

    value, delta, gamma, vega, rho, theta = american_option_with_greeks(
        option_type, S, K, T, r, q, sigma, steps=500
    )

    results.append([option_id, value, delta, gamma, vega, rho, theta])

out = pd.DataFrame(
    results,
    columns=["ID", "Value", "Delta", "Gamma", "Vega", "Rho", "Theta"]
)

print(out.to_string(index=False))

 ID     Value     Delta    Gamma      Vega        Rho     Theta
  1  3.259347  0.547849 0.055326 14.651750  -0.446486 13.014966
  2  2.692775 -0.470676 0.057210 14.611928  -0.253716  8.940220
  3 22.050266  0.676370 0.010698 35.904831 -22.896127  7.280164
  4 21.019233 -0.489952 0.002654 40.759301 -14.102243  5.956521
